# 실기 대비
# 실전 문제풀이
# set 4

## 1) 데이터 및 시나리오

### 디지털프라자 매출 데이터

> 디지털프라자 A지점에서 최근 연휴 직전에 재고 처리와 매출 신장을 위해 대대적인 할인 행사를 하였다.  
> 하루만 반짝 진행한 행사에서 예상보다 많은 손님이 방문했고 이번에 발생한 매출데이터를 취합하여 향후 발송할 판촉물에 들어갈 컨텐츠를 기획하고자 한다.  
> 취합한 데이터는 다음과 같고 한 개의 행이 물품 1개 구매 내역이다.

### 데이터 개요

| 파일명 | 행 | 열 | 인코딩 |
|---|---:|---:|---|
| `sales_pos.csv` | 550068 | 11 | UTF-8 |

## 1) 데이터 및 시나리오

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `user` | int | 고객 식별자 |
| `prod` | string | 상품 식별자 |
| `gender` | string | 성별 |
| `age_group` | string | 연령대 |
| `job` | int | 직업 구분 |
| `city` | string | 도시 유형 구분 |
| `marital` | int | 결혼 여부 `(1: 결혼)` |
| `prod_cat1` | int | 상품 카테고리 `(1차)` |
| `prod_cat2` | int | 상품 카테고리 `(2차)` |
| `prod_cat3` | int | 상품 카테고리 `(3차)` |
| `purchase` | int | 결제 금액 |

## 2) 문제

### 필요 라이브러리 함수 및 클래스 목록

| 목록 |
|---|
| `from sklearn.preprocessing import MinMaxScaler` |
| `from sklearn.cluster import KMeans` |
| `from sklearn.metrics import silhouette_score` |


In [134]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv("../dataset/sales_pos.csv")
display(df.shape)
display(df.dtypes)
display(df.isna().sum())
display(df)

(550068, 11)

user           int64
prod          object
gender        object
age_group     object
job            int64
city          object
marital        int64
prod_cat1      int64
prod_cat2    float64
prod_cat3    float64
purchase       int64
dtype: object

user              0
prod              0
gender            0
age_group         0
job               0
city              0
marital           0
prod_cat1         0
prod_cat2    173638
prod_cat3    383247
purchase          0
dtype: int64

,user,prod,gender,age_group,job,city,marital,prod_cat1,prod_cat2,prod_cat3,purchase
0,1,P00069042,F,0-17,10,A,0,3,NaN,NaN,8370
1,1,P00248942,F,0-17,10,A,0,1,6.0,14.0,15200
2,1,P00087842,F,0-17,10,A,0,12,NaN,NaN,1422
3,1,P00085442,F,0-17,10,A,0,12,14.0,NaN,1057
4,2,P00285442,M,55+,16,C,0,8,NaN,NaN,7969
...,...,...,...,...,...,...,...,...,...,...,...
550063,6033,P00372445,M,51-55,13,B,1,20,NaN,NaN,368
550064,6035,P00375436,F,26-35,1,C,0,20,NaN,NaN,371
550065,6036,P00375436,F,26-35,15,B,1,20,NaN,NaN,137
550066,6038,P00375436,F,55+,1,C,0,20,NaN,NaN,365



### Q01.

상품별 매출액(`purchase`)을 합산하여 그 매출액이 가장 큰 상품을 확인하고 해당 상품을 가장 많이 구매하는 직업(`job`)을 확인하시오.

※ 분석 결과를 기반으로 `job` 변수의 번호를 최종 출력하시오.  
※ 직업 확인시 상품 구매 개수를 기준으로 확인하시오. `(정답 예시: 1)`

In [135]:
df_q1 = df.copy()
display(df_q1['prod'].value_counts())

df_q1_gb = df_q1.groupby('prod')['purchase'].sum()
max_purchase = df_q1_gb.idxmax()
display(max_purchase)

df_q1_2 = df_q1.loc[df_q1['prod'] == max_purchase, :]
df_q1_2['job'].value_counts().idxmax()

P00265242    1880
P00025442    1615
P00110742    1612
P00112142    1562
P00057642    1470
             ... 
P00314842       1
P00298842       1
P00231642       1
P00204442       1
P00066342       1
Name: prod, Length: 3631, dtype: int64

'P00025442'

4

### Q02.

결혼 여부(`marital`)에 따라 구매하는 물품의 종류가 많이 차이 나는지 확인하고자 한다.  
비교적 신혼부부가 많은 26-35세 그룹을 대상으로 각 고객의 구매물품 카테고리 개수를 산출하고 결혼여부별 그 평균값의 차이를 산출하시오.

※ 구매 물품의 카테고리 개수 산출에는 `prod_cat1`, `prod_cat2`, `prod_cat3` 변수를 사용한다.  
※ 카테고리 관련 변수의 결측치는 0으로 대체한다.  
※ 결측치를 대체한 데이터까지 포함하여 문제를 풀이하시오.  
※ 카테고리 관련 변수의 처리 예시는 다음과 같다.

| `prod_cat1` | `prod_cat2` | `prod_cat3` | `prod_cat` |
|---:|---:|---:|---|
| 1 | 2 | 0 | `1-2-0` |
| 8 | 0 | 0 | `8-0-0` |

※ 고객 식별자가 1인 고객은 총 21개 카테고리의 물품을 구매하였다.  
※ 정답은 절대값을 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [ ]:
df_q2 = df.copy()
display(df_q2['age_group'].value_counts())
df_q2_26_35 = df_q2.loc[df_q2['age_group'].isin(['26-35']), : ].copy()
display(df_q2_26_35.shape[0])

display(df_q2_26_35.isna().any())
df_q2_26_35[['prod_cat2','prod_cat3']] = df_q2_26_35[['prod_cat2','prod_cat3']].fillna(0)
display(df_q2_26_35.isna().any())


df_q2_2 = df_q2_26_35.copy()
df_q2_2[['prod_cat1','prod_cat2','prod_cat3']] = df_q2_2[['prod_cat1','prod_cat2','prod_cat3']].astype(int).astype(str)
display(df_q2_2['prod_cat1'].dtype,df_q2_2['prod_cat2'].dtype,df_q2_2['prod_cat3'].dtype)
df_q2_2['prod_cat'] = df_q2_2['prod_cat1'] + '-' + df_q2_2['prod_cat2'] + '-' + df_q2_2['prod_cat3']
display(df_q2_2)

#결혼 여부 별 각 고객의 구매물품카테고리 개수 평균(user-marital이 한가지 case만 되므로 중요한 포인트)
df_q2_2_gb = df_q2_2.groupby(['user','marital'])['prod_cat'].nunique().reset_index(name = 'prod_cat_count')
display(df_q2_2_gb)

df_q2_2_gb2 = df_q2_2_gb.groupby('marital')['prod_cat_count'].mean()
display(df_q2_2_gb2)

round(abs(df_q2_2_gb2[0] - df_q2_2_gb2[1]),2)

26-35    219587
36-45    110013
18-25     99660
46-50     45701
51-55     38501
55+       21504
0-17      15102
Name: age_group, dtype: int64

219587

user         False
prod         False
gender       False
age_group    False
job          False
city         False
marital      False
prod_cat1    False
prod_cat2     True
prod_cat3     True
purchase     False
dtype: bool

user         False
prod         False
gender       False
age_group    False
job          False
city         False
marital      False
prod_cat1    False
prod_cat2    False
prod_cat3    False
purchase     False
dtype: bool

dtype('O')

...

dtype('O')

,user,prod,gender,age_group,job,city,marital,prod_cat1,prod_cat2,prod_cat3,purchase,prod_cat
5,3,P00193542,M,26-35,15,A,0,1,2,0,15227,1-2-0
9,5,P00274942,M,26-35,20,A,1,8,0,0,7871,8-0-0
10,5,P00251242,M,26-35,20,A,1,5,11,0,5254,5-11-0
11,5,P00014542,M,26-35,20,A,1,8,0,0,3957,8-0-0
12,5,P00031342,M,26-35,20,A,1,8,0,0,6073,8-0-0
...,...,...,...,...,...,...,...,...,...,...,...,...
550058,6024,P00372445,M,26-35,12,A,1,20,0,0,121,20-0-0
550059,6025,P00370853,F,26-35,1,B,0,19,0,0,48,19-0-0
550061,6029,P00372445,F,26-35,1,C,1,20,0,0,599,20-0-0
550064,6035,P00375436,F,26-35,1,C,0,20,0,0,371,20-0-0


,user,marital,prod_cat_count
0,3,0,18
1,5,1,43
2,8,1,32
3,9,0,31
4,11,0,34
...,...,...,...
2048,6030,1,33
2049,6034,0,8
2050,6035,0,61
2051,6036,1,123


marital
0    41.663183
1    41.792336
Name: prod_cat_count, dtype: float64

0.13

### Q03.

고객 5891명을 군집화 하여 각 군집별로 마케팅 전략을 수립하고자 한다.  
다음에 제시된 변수를 대상으로 k-means 군집분석을 실시하고 7개 군집으로 분석했을 때 Silhouette score를 산출하시오.

#### 독립변수

| 독립변수 |
|---|
| 성별 |
| 나이 |
| 직업 |
| 도시 |
| 결혼 여부 |
| 구매 상품 종류수 |
| 총 구매금액 |

※ 구매 상품 종류수 변수는 `prod` 변수를 참고하여 생성하시오.  
※ 성별 변수는 `gender` 변수에서 `M`을 1, `F`를 0으로 변환하여 사용하시오.  
※ 나이는 `age_group` 변수에서 나이가 가장 적은 그룹을 0으로 지정하고 가장 나이가 많은 그룹은 6으로 지정하는 방식으로 순서형 변수로 변환하시오.  
※ 직업과 도시 변수는 One Hot Encoding 변환하여 사용하시오.  
※ 군집 분석에 사용되는 변수는 총 29개이며 MinMax 정규화 후 분석하시오.  
※ seed는 `123`으로 지정하시오.  
※ 결과는 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [ ]:
df_q3 = df.copy()
df_q3_user = df_q3.groupby('user').agg(
    gender=('gender', 'first'),
    age_group=('age_group','first'),
    job=('job','first'),
    city=('city','first'),
    marital=('marital','first'),
    prod=('prod','nunique'),
    purchase=('purchase','sum')
)#groupby로 'user'가 index가 되니 'user'도 columns 으로 쓰지 않기 위해 reset_index 안함

display(df_q3_user)

display('------------------------------')
display(df_q3_user['gender'].value_counts())
df_q3_user['gender'] = df_q3_user['gender'].replace({"M" : 1,"F" : 0})
display(df_q3_user['gender'].value_counts())


display('------------------------------')
# df_q3_user['age_group'].drop_duplicates() = pd.Series(df_q3_user['age_group'].unique(), name='age_group')
ser = df_q3_user['age_group'].drop_duplicates().sort_values(ascending = True).reset_index(drop=True)
display(type(ser), ser)
age_group_dict = dict(zip(ser, ser.index)) #반복문 없이 한번에하는게 zip
display(type(age_group_dict), age_group_dict)

display(df_q3_user['age_group'].value_counts())
df_q3_user['age_group'] = df_q3_user['age_group'].replace(age_group_dict)
display(df_q3_user['age_group'].value_counts())
display('------------------------------')

df_q3_base = pd.get_dummies(df_q3_user, columns = ['job','city'])
display(df_q3_base.shape)


,gender,age_group,job,city,marital,prod,purchase
user,,,,,,,
1,F,0-17,10,A,0,35,334093
2,M,55+,16,C,0,77,810472
3,M,26-35,15,A,0,29,341635
4,M,46-50,7,B,1,14,206468
5,M,26-35,20,A,1,106,821001
...,...,...,...,...,...,...,...
6036,F,26-35,15,B,1,514,4116058
6037,F,46-50,1,C,0,122,1119538
6038,F,55+,1,C,0,12,90034


'------------------------------'

M    4225
F    1666
Name: gender, dtype: int64

1    4225
0    1666
Name: gender, dtype: int64

'------------------------------'

...

{'0-17': 0,
 '18-25': 1,
 '26-35': 2,
 '36-45': 3,
 '46-50': 4,
 '51-55': 5,
 '55+': 6}

26-35    2053
36-45    1167
18-25    1069
46-50     531
51-55     481
55+       372
0-17      218
Name: age_group, dtype: int64

2    2053
3    1167
1    1069
4     531
5     481
6     372
0     218
Name: age_group, dtype: int64

'------------------------------'

(5891, 29)

In [138]:
#D
X = df_q3_base.copy()

#N
scaler = MinMaxScaler()
X_n = scaler.fit_transform(X)
#M
model = KMeans(
    n_clusters = 7,
    random_state = 123
)
y_pred = model.fit_predict(X_n)
#E
round(silhouette_score(X_n, y_pred) ,2)

0.18